<a href="https://colab.research.google.com/github/mancinigabriel/tcc-pece-assin2-llm-challenges/blob/main/notebooks/Ministral3_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/mancinigabriel/tcc-pece-assin2-llm-challenges.git

fatal: destination path 'tcc-pece-assin2-llm-challenges' already exists and is not an empty directory.


#Preparando ambiente

In [ ]:
!pip install transformers==5.0.0rc0
!pip install mistral-common

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath('/content/tcc-pece-assin2-llm-challenges')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend, FineGrainedFP8Config
from datetime import datetime, timezone, timedelta
from src import prompts
from src import utils
from src import data
from src import metrics
import pandas as pd
import random
import torch
import re
import time

In [ ]:
cfg = utils.load_config(
    "/content/tcc-pece-assin2-llm-challenges/configs/base.yaml",
    "/content/tcc-pece-assin2-llm-challenges/configs/models/mistral3.yaml"
)

generation_args = cfg["generation"]

gpu = 'L4'

#Ministral 3 - 3B - Instruct

In [ ]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "mistralai/Ministral-3-3B-Instruct-2512"
model_name = "mistral3b_instruct_fp8"
quantizado = True
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


Loading weights:   0%|          | 0/822 [00:00<?, ?it/s]

##Teste de Consistência

In [ ]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:58:06
100 - 2026-01-11 23:58:39
200 - 2026-01-11 23:59:12


##Loop de aplicação do prompt em todo o dataset

In [ ]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

In [ ]:
timing = {}

timing['inicio'] = utils.time_log()
timing['inicio_cons'] = utils.time_log()
timing['inicio_aplicacao_total'] = utils.time_log()

df_sintetico = data.gera_df_sintetico()

for i in range(len(df_sintetico)):
  premissa = df_sintetico.iloc[i]['premissa']
  hipotese = df_sintetico.iloc[i]['hipotese']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_sintetico.loc[i, column_name] = resp

df_sintetico[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_sintetico[column_name]))

timing['fim_cons'] = utils.time_log()
timing['fim'] = utils.time_log()

metrics_dict = {'acurácia_df_sintetico': metrics.calculate_accuracy(df_sintetico)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, f"{model_name}_df_sintetico")
df_sintetico.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_df_sintetico.csv')

# Mistral 3 - 8B - Instruct

In [ ]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "mistralai/Ministral-3-8B-Instruct-2512"
model_name = "mistral8b_instruct_fp8"
quantizado = True
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

In [ ]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

In [ ]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

In [ ]:
timing = {}

timing['inicio'] = utils.time_log()
timing['inicio_cons'] = utils.time_log()
timing['inicio_aplicacao_total'] = utils.time_log()

df_sintetico = data.gera_df_sintetico()

for i in range(len(df_sintetico)):
  premissa = df_sintetico.iloc[i]['premissa']
  hipotese = df_sintetico.iloc[i]['hipotese']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_sintetico.loc[i, column_name] = resp

df_sintetico[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_sintetico[column_name]))

timing['fim_cons'] = utils.time_log()
timing['fim'] = utils.time_log()

metrics_dict = {'acurácia_df_sintetico': metrics.calculate_accuracy(df_sintetico)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, f"{model_name}_df_sintetico")
df_sintetico.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_df_sintetico.csv')